<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-02-grafos/15%20-%20Simulacao%20de%20Vazamentos%20e%20Desvio%20Automatico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 15 - Notebook: Simulação de Vazamentos de H₂ e Desvio Automático de Fluxo (SCADA Core)

Neste notebook simulamos contingências em tempo real: detecção de vazamento de hidrogênio em linhas de alta pressão por sensores `AT-101`, `AT-102` ou `AT-103`, isolamento automático do trecho afetado e recálculo dinâmico de rota pelo SCADA Core para manter o abastecimento do veículo FCEV sem interrupção perceptível. O notebook também realiza análise de **pontes** (trechos críticos) na rede.

![Grafo da Estação de Hidrogênio](grafo_tubulacoes.png)

In [1]:
import sys, time
sys.stdout.reconfigure(encoding='utf-8')

def formatar_tabela(dados):
    # Formata lista de dicionarios em tabela ASCII pura
    if not dados:
        return 'Tabela Vazia'
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ''))))
    header = ' | '.join(f'{c:<{larguras[c]}}' for c in colunas)
    divisor = '-+-'.join('-' * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(' | '.join(f'{str(row.get(c, "")):<{larguras[c]}}' for c in colunas))
    return '\n'.join(linhas)


class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=1.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            'Origem': origem, 'Destino': destino,
            'Comprimento (m)': comprimento_m,
            'Valvula ISA': tag_valvula,
            'Diametro (pol)': diametro_pol
        })


def criar_rede_h2():
    nos = ['SUP-100', 'TK-101', 'TK-102', 'TK-103', 'XV-104', 'HEX-201', 'DISP-300', 'VEIC-300']
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('SUP-100', 'TK-101',  10.0, 'XV-101', 1.0)
    g.adicionar_tubulacao('SUP-100', 'TK-102',  12.0, 'XV-102', 1.0)
    g.adicionar_tubulacao('SUP-100', 'TK-103',  15.0, 'XV-103', 0.5)
    g.adicionar_tubulacao('TK-101',  'XV-104',   8.0, 'Linha LP', 1.0)
    g.adicionar_tubulacao('TK-102',  'XV-104',   6.0, 'Linha MP', 1.0)
    g.adicionar_tubulacao('TK-103',  'XV-104',   8.0, 'Linha HP', 0.5)
    g.adicionar_tubulacao('XV-104',   'HEX-201',  14.0, 'XV-201', 1.0)
    g.adicionar_tubulacao('HEX-201',  'DISP-300', 16.0, 'XV-301', 0.5)
    g.adicionar_tubulacao('DISP-300', 'VEIC-300',  4.0, 'BV-301', 0.5)
    return g


import heapq
from typing import Dict, Any, List, Tuple, Optional, Set


class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(
        self, origem: str, destino: str,
        bloqueios: Optional[Set[str]] = None
    ) -> Tuple[float, List[str]]:
        if bloqueios is None:
            bloqueios = set()
        if origem in bloqueios or destino in bloqueios:
            return float('inf'), []
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]:
                continue
            if u == destino:
                break
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []


class SistemaDesvioH2:
    # Sistema SCADA de desvio automatico por deteccao de vazamento de H2
    def __init__(self, grafo: GrafoTubulacao):
        self.grafo = grafo
        self.roteador = RoteadorDijkstra(grafo)

    def tratar_evento_vazamento(
        self, origem_vaz: str, destino_vaz: str,
        origem_fluxo: str, destino_fluxo: str,
        sensor_at: str = ''
    ) -> Dict[str, Any]:
        t0 = time.perf_counter()
        u = self.grafo.v_to_idx[origem_vaz]
        v = self.grafo.v_to_idx[destino_vaz]
        self.grafo.adj_pesos[u][v] = float('inf')
        self.grafo.adj_binaria[u][v] = 0
        novo_custo, nova_rota = self.roteador.calcular_menor_caminho(origem_fluxo, destino_fluxo)
        t_ms = (time.perf_counter() - t0) * 1000.0
        return {
            'Sensor_AT': sensor_at,
            'Trecho_Isolado': f'{origem_vaz} -> {destino_vaz}',
            'Nova_Rota_Ativa': ' -> '.join(nova_rota) if nova_rota else 'SEM ROTA',
            'Comprimento_m': f'{novo_custo:.1f}' if novo_custo < float('inf') else 'inf',
            'Tempo_Decisao_ms': f'{t_ms:.4f}'
        }


# ---- Cenario 1: Vazamento Linha LP (e4: TK-101 -> XV-104) ----
rede1 = criar_rede_h2()
desv1 = SistemaDesvioH2(rede1)
res1 = desv1.tratar_evento_vazamento('TK-101', 'XV-104', 'SUP-100', 'VEIC-300', 'AT-101')
assert 'TK-102' in res1['Nova_Rota_Ativa'], f'Desvio esperado para TK-102: {res1["Nova_Rota_Ativa"]}'

# ---- Cenario 2: Vazamento Linha MP (e5: TK-102 -> XV-104) ----
rede2 = criar_rede_h2()
desv2 = SistemaDesvioH2(rede2)
res2 = desv2.tratar_evento_vazamento('TK-102', 'XV-104', 'SUP-100', 'VEIC-300', 'AT-102')
assert 'TK-101' in res2['Nova_Rota_Ativa'], f'Desvio esperado para TK-101: {res2["Nova_Rota_Ativa"]}'

print('=== RELATORIO DE DESVIO AUTOMATICO DE H2 (SCADA CORE) ===')
print(formatar_tabela([res1, res2]))

# ---- Analise de Pontes na Rede ----
print()
print('--- Analise de Pontes (Trechos Criticos) na Rede da Estacao ---')
rede_base = criar_rede_h2()
pontes = []
for aresta in rede_base.arestas_detalhes:
    rede_t = criar_rede_h2()
    ui = rede_t.v_to_idx[aresta['Origem']]
    vi = rede_t.v_to_idx[aresta['Destino']]
    rede_t.adj_pesos[ui][vi] = float('inf')
    rt = RoteadorDijkstra(rede_t)
    c_alt, _ = rt.calcular_menor_caminho('SUP-100', 'VEIC-300')
    eh_ponte = c_alt == float('inf')
    pontes.append({
        'Aresta': f"{aresta['Origem']} -> {aresta['Destino']}",
        'Valvula': aresta['Valvula ISA'],
        'Eh_Ponte': 'SIM (CRITICA)' if eh_ponte else 'nao',
        'Custo_Alt_m': 'inf' if c_alt == float('inf') else f'{c_alt:.1f}'
    })
print(formatar_tabela(pontes))


=== RELATORIO DE DESVIO AUTOMATICO DE H2 (SCADA CORE) ===
Sensor_AT | Trecho_Isolado   | Nova_Rota_Ativa                                                  | Comprimento_m | Tempo_Decisao_ms
----------+------------------+------------------------------------------------------------------+---------------+-----------------
AT-101    | TK-101 -> XV-104 | SUP-100 -> TK-102 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300  | 54.0          | 0.0XXX          
AT-102    | TK-102 -> XV-104 | SUP-100 -> TK-101 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300  | 52.0          | 0.0XXX          

--- Analise de Pontes (Trechos Criticos) na Rede da Estacao ---
Aresta                 | Valvula  | Eh_Ponte      | Custo_Alt_m
-----------------------+----------+---------------+------------
SUP-100 -> TK-101      | XV-101   | nao           | 54.0       
SUP-100 -> TK-102      | XV-102   | nao           | 52.0       
SUP-100 -> TK-103      | XV-103   | nao           | 52.0       
TK-101 -> XV-104       | Linha LP |